In [8]:
import re, math, random, warnings
import numpy as np
import pandas as pd
import nltk, spacy, torch
import torch.nn as nn

from collections import Counter

from nltk import *
from nltk.stem import *
from nltk.corpus import *
from nltk.wsd import *
from nltk.lm import *
from nltk.lm.preprocessing import *
from nltk.translate.bleu_score import *

from sklearn.model_selection import train_test_split, cross_val_score, cross_validate
from sklearn.feature_extraction.text import *
from sklearn.metrics import *
from sklearn.metrics.pairwise import *
from sklearn.preprocessing import LabelEncoder
from sklearn.neural_network import MLPClassifier
import spacy
nlp = spacy.load("en_core_web_sm")

from gensim.models import *
from transformers import *

In [9]:
df = pd.read_csv("data1.csv")

In [10]:
text = str(df["text"][0])
tokens = word_tokenize(text.lower())

print("Tokens:", tokens)
print("Stems:", [PorterStemmer().stem(w) for w in tokens])
print("Lemmas:", [WordNetLemmatizer().lemmatize(w) for w in tokens])

Tokens: ['i', 'love', 'this', 'phone', 'and', 'its', 'amazing', 'battery']
Stems: ['i', 'love', 'thi', 'phone', 'and', 'it', 'amaz', 'batteri']
Lemmas: ['i', 'love', 'this', 'phone', 'and', 'it', 'amazing', 'battery']


In [11]:
uni, bi = Counter(tokens), Counter(zip(tokens[:-1], tokens[1:]))
w1, w2 = tokens[0], tokens[1]
V = len(set(tokens))
print("Bigram probability:", bi[(w1, w2)] / uni[w1])
print("Laplace smoothing:", (bi[(w1, w2)] + 1) / (uni[w1] + V))

Bigram probability: 1.0
Laplace smoothing: 0.2222222222222222


In [12]:
# BPE: merge the most frequent adjacent character pair in common dataset words
pairs = Counter()
for word, count in Counter(tokens).most_common(20):
    pairs.update(zip(word, word[1:]))
pair = pairs.most_common(1)[0][0]
print("BPE merge:", pair, "->", "".join(pair))

# Tiny neural language model: previous word predicts next word
word_pairs = list(zip(tokens[:-1], tokens[1:]))
xe, ye = LabelEncoder(), LabelEncoder()
X = xe.fit_transform([a for a, b in word_pairs]).reshape(-1, 1)
y = ye.fit_transform([b for a, b in word_pairs])
model = MLPClassifier(hidden_layer_sizes=(8,), max_iter=300, random_state=1).fit(X, y)
print("Next word after", tokens[0], ":",
      ye.inverse_transform(model.predict(xe.transform([tokens[0]]).reshape(-1, 1)))[0])

BPE merge: ('l', 'o') -> lo
Next word after i : love


C:\Users\nikil\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (300) reached and the optimization hasn't converged yet.
  warnings.warn(


In [13]:
print(pos_tag(tokens[:15]))

[('i', 'RB'), ('love', 'VBP'), ('this', 'DT'), ('phone', 'NN'), ('and', 'CC'), ('its', 'PRP$'), ('amazing', 'JJ'), ('battery', 'NN')]


In [18]:
cfg = CFG.fromstring("""
S -> NP VP
NP -> Det N | Det N PP
VP -> V NP | VP PP
PP -> P NP
Det -> 'the'
N -> 'boy' | 'man' | 'telescope'
V -> 'saw'
P -> 'with'
""")

parser = ChartParser(cfg)
sentence = "the boy saw the man with the telescope".split()

for tree in parser.parse(sentence):
    tree.pretty_print()

                 S                                
      ___________|_______                          
     |                   VP                       
     |            _______|________                 
     |           VP               PP              
     |        ___|___         ____|___             
     NP      |       NP      |        NP          
  ___|___    |    ___|___    |     ___|______      
Det      N   V  Det      N   P   Det         N    
 |       |   |   |       |   |    |          |     
the     boy saw the     man with the     telescope

                 S                            
      ___________|___                          
     |               VP                       
     |        _______|___                      
     |       |           NP                   
     |       |    _______|____                 
     |       |   |   |        PP              
     |       |   |   |    ____|___             
     NP      |   |   |   |        NP          
  ___|

In [19]:
pcfg = PCFG.fromstring("""
S -> NP VP [1.0]
NP -> Det N [0.6] | Det N PP [0.4]
VP -> V NP [0.7] | VP PP [0.3]
PP -> P NP [1.0]
Det -> 'the' [1.0]
N -> 'boy' [0.33] | 'man' [0.33] | 'telescope' [0.34]
V -> 'saw' [1.0]
P -> 'with' [1.0]
""")

parser = ViterbiParser(pcfg)

for tree in parser.parse(sentence):
    tree.pretty_print()
    print("Probability:", tree.prob())

                 S                            
      ___________|___                          
     |               VP                       
     |        _______|___                      
     |       |           NP                   
     |       |    _______|____                 
     |       |   |   |        PP              
     |       |   |   |    ____|___             
     NP      |   |   |   |        NP          
  ___|___    |   |   |   |     ___|______      
Det      N   V  Det  N   P   Det         N    
 |       |   |   |   |   |    |          |     
the     boy saw the man with the     telescope

Probability: 0.0037322208000000003


In [21]:
text = str(df["text"][0])
doc = nlp(text)

for token in doc:
    print(f"{token.text:12} --({token.dep_:8})--> {token.head.text}")

I            --(nsubj   )--> love
love         --(ROOT    )--> love
this         --(det     )--> phone
phone        --(dobj    )--> love
and          --(cc      )--> phone
its          --(poss    )--> battery
amazing      --(amod    )--> battery
battery      --(conj    )--> phone


In [23]:
syn = wn.synsets(tokens[3])
print(syn[0].definition() if syn else "No WordNet meaning found")

electronic equipment that converts sound into electrical signals that can be transmitted over distances and then converts received signals back into sounds


In [29]:
texts = df["text"].dropna().astype(str).tolist()

X = TfidfVectorizer().fit_transform(texts)

print("TF-IDF:", X.toarray().round(2))

if X.shape[0] >= 2:
    print("Cosine similarity:", cosine_similarity(X[0], X[1])[0, 0])
    print("Euclidean distance:", euclidean_distances(X[0], X[1])[0, 0])

TF-IDF: [[0.   0.   0.45 ... 0.   0.   0.  ]
 [0.   0.   0.   ... 0.   0.   0.  ]
 [0.   0.   0.   ... 0.   0.5  0.  ]
 ...
 [0.43 0.   0.   ... 0.   0.   0.  ]
 [0.   0.   0.   ... 0.   0.   0.  ]
 [0.   0.   0.   ... 0.   0.   0.  ]]
Cosine similarity: 0.0
Euclidean distance: 1.4142135623730951


In [30]:
# 1. Evaluation Metrics (4.4)

# 1. BLEU Score for Machine Translation / Generation
ref = [["the", "cat", "sat", "on", "the", "mat"]]
cand = ["the", "cat", "is", "on", "the", "mat"]
print("BLEU Score:", round(sentence_bleu(ref, cand), 4))

# 2. Classification Metrics
y_true, y_pred = [1, 0, 1, 1], [1, 0, 0, 1]
print("Accuracy :", accuracy_score(y_true, y_pred))
print("Precision:", precision_score(y_true, y_pred))
print("Recall   :", recall_score(y_true, y_pred))
print("F1 Score :", round(f1_score(y_true, y_pred), 3))

BLEU Score: 0.0
Accuracy : 0.75
Precision: 1.0
Recall   : 0.6666666666666666
F1 Score : 0.8


C:\Users\nikil\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\nltk\translate\bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 4-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(


In [31]:
print("Text:", texts[0])
print("Coreference task: identify which earlier noun a pronoun (he/she/it/they) refers to.")
print("This is a concept demonstration; a real resolver needs a coreference model.")

Text: I love this phone and its amazing battery
Coreference task: identify which earlier noun a pronoun (he/she/it/they) refers to.
This is a concept demonstration; a real resolver needs a coreference model.


In [32]:
for ent in nlp(texts[0]).ents:
    print(ent.text, "->", ent.label_)

sentiment = pipeline("sentiment-analysis")  # first use may download model
print(sentiment(texts[0]))

[transformers] No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.
[transformers] loading configuration file config.json from cache at C:\Users\nikil\.cache\huggingface\hub\models--distilbert--distilbert-base-uncased-finetuned-sst-2-english\snapshots\714eb0fa89d2f80546fda750413ed43d93601a13\config.json
[transformers] Model config DistilBertConfig {
  "activation": "gelu",
  "architectures": [
    "DistilBertForSequenceClassification"
  ],
  "attention_dropout": 0.1,
  "bos_token_id": null,
  "dim": 768,
  "dropout": 0.1,
  "eos_token_id": null,
  "finetuning_task": "sst-2",
  "hidden_dim": 3072,
  "id2label": {
    "0": "NEGATIVE",
    "1": "POSITIVE"
  },
  "initializer_range": 0.02,
  "label2id": {
    "NEGATIVE": 0,
    "POSITIVE": 1
  },
  "max_position_embeddings": 512,
  "model_type": "distilbert",
  "n_heads": 12,
  "n_l

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

[transformers] loading configuration file config.json from cache at C:\Users\nikil\.cache\huggingface\hub\models--distilbert--distilbert-base-uncased-finetuned-sst-2-english\snapshots\714eb0fa89d2f80546fda750413ed43d93601a13\config.json
[transformers] Model config DistilBertConfig {
  "activation": "gelu",
  "architectures": [
    "DistilBertForSequenceClassification"
  ],
  "attention_dropout": 0.1,
  "bos_token_id": null,
  "dim": 768,
  "dropout": 0.1,
  "eos_token_id": null,
  "finetuning_task": "sst-2",
  "hidden_dim": 3072,
  "id2label": {
    "0": "NEGATIVE",
    "1": "POSITIVE"
  },
  "initializer_range": 0.02,
  "label2id": {
    "NEGATIVE": 0,
    "POSITIVE": 1
  },
  "max_position_embeddings": 512,
  "model_type": "distilbert",
  "n_heads": 12,
  "n_layers": 6,
  "output_past": true,
  "pad_token_id": 0,
  "qa_dropout": 0.1,
  "seq_classif_dropout": 0.2,
  "sinusoidal_pos_embds": false,
  "tie_weights_": true,
  "tie_word_embeddings": true,
  "transformers_version": "5.17.0"

[{'label': 'POSITIVE', 'score': 0.9998812675476074}]
